# Iteration 0: Baseline Binary Classification (Process Safety vs Non-Process Safety) - RQ1

## 1. Configuration & Data Loading
Set up cross-platform paths, define output directories, and load four JSONL source files into DataFrames.

In [35]:
# -----------------------------
# Configuration & Path Setups
# -----------------------------

import pandas as pd
import numpy as np
import os
from pathlib import Path
from typing import Union, Optional
import plotly.graph_objects as go

# Cross-platform path resolution (consistent with Iteration 0)
def find_project_root_with_datasets(start_path: Path, max_levels: int = 10) -> Path:
    """
    Search upward from start_path for a directory containing 'Datasets' folder.
    This makes the notebook work on any system (macOS, Windows, Linux).
    """
    cur = start_path.resolve()
    for _ in range(max_levels):
        if (cur / 'Datasets').exists():
            return cur
        cur = cur.parent
    raise FileNotFoundError(
        f"Could not find project root with 'Datasets' folder within {max_levels} levels. "
        f"Set THESIS_BASE_DIR environment variable or ensure Datasets folder exists."
    )

# Try environment variable first, then search for project root
env_base = os.environ.get('THESIS_BASE_DIR')
if env_base:
    BASE_DIR = Path(env_base)
    print(f"Using THESIS_BASE_DIR from environment: {BASE_DIR}")
else:
    BASE_DIR = find_project_root_with_datasets(Path.cwd())
    print(f"Found project root: {BASE_DIR}")

PATHS = {
    'raw_data': BASE_DIR / "Datasets" / "RW_Datasets",
    'iteration_output': BASE_DIR / "Datasets" / "Iteration_Outputs" / "_iteration_0",
    'embeddings': BASE_DIR / "Datasets" / "Embeddings" / "_iteration_0" / "bert-base-uncased",
    'results': BASE_DIR / "Results" / "_iteration_0"
}

# Normalize all paths to absolute Path objects
for k, p in list(PATHS.items()):
    PATHS[k] = Path(p).resolve()

# Create output directories
for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

# Data files configuration (JSONL files)
DATA_FILES = {
    'df_14': PATHS['raw_data'] / "14K_Reports_RW_ACTUALS_PS.jsonl",
    'df_28': PATHS['raw_data'] / "28k_Reports_RW_ACTUALS_ALL.jsonl",
    'df_30': PATHS['raw_data'] / "30K_Reports_RW_ACTUALS_ALL_v2.0.jsonl",
    'df_700': PATHS['raw_data'] / "700k_reports_RW_ACTUALS_NH_OBS.jsonl"
}

def load_data(file_path: Union[str, Path]) -> Optional[pd.DataFrame]:
    """
    Load a JSONL file into a DataFrame with error handling.
    
    Parameters: 
        file_path (Union[str, Path]): Path to the JSONL data file
        
    Returns:
        pandas.DataFrame: Loaded data or None if file not found
    """
    file_path = Path(file_path)
    try:
        df = pd.read_json(file_path, lines=True)
        print(f"Successfully loaded {file_path.name} ({len(df)} rows)")
        return df
    except Exception as e:
        print(f"An error occurred while loading {file_path.name}: {e}")
        return None

# Load Each File (JSONL files)
RW_ACTUALS_PS_14 = load_data(DATA_FILES['df_14'])
RW_ACTUALS_ALL_28 = load_data(DATA_FILES['df_28'])
RW_ACTUALS_ALL_30 = load_data(DATA_FILES['df_30'])
RW_ACTUALS_NH_OBS_700 = load_data(DATA_FILES['df_700'])

# Check if all dataframes were loaded successfully
if all(df is not None for df in [RW_ACTUALS_PS_14, RW_ACTUALS_ALL_28, RW_ACTUALS_ALL_30, RW_ACTUALS_NH_OBS_700]):
    print("\n[OK] All files loaded successfully!")
else:
    print("\n[WARNING] Some files failed to load. Please check the file paths and try again.")

Found project root: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026
Successfully loaded 14K_Reports_RW_ACTUALS_PS.jsonl (14432 rows)
Successfully loaded 28k_Reports_RW_ACTUALS_ALL.jsonl (28323 rows)
Successfully loaded 30K_Reports_RW_ACTUALS_ALL_v2.0.jsonl (30609 rows)
Successfully loaded 700k_reports_RW_ACTUALS_NH_OBS.jsonl (717041 rows)

[OK] All files loaded successfully!


## 2. Schema Alignment
Rename and drop columns to align all four datasets to a common schema. The 700K dataset renames `DATETIME` → `CASE_OCCURENCE_DATE`. The 30K dataset renames two columns, adds a missing column (`IMM_ACTION_TAKEN_RECOM`), and drops seven extra columns.

In [36]:
# =============================================================================
# Adjust datasets according to the column mapping comments
# =============================================================================

# --- 700K Dataset Fixes ---
# Rename DATETIME → CASE_OCCURENCE_DATE (comment: "Rename in 700k Reports")
rename_map_700k = {
    "DATETIME": "CASE_OCCURENCE_DATE",
}
RW_ACTUALS_NH_OBS_700 = RW_ACTUALS_NH_OBS_700.rename(
    columns={k: v for k, v in rename_map_700k.items() if k in RW_ACTUALS_NH_OBS_700.columns}
)

# --- 30K Dataset Fixes ---
# 1) Rename columns (comment: "Rename in 30k Reports")
rename_map_30k = {
    "EMPLOYMENT_CATEGORY": "COMPANY_INVOLVED_TYPE",
    "HAZARD_PHYSICAL_SECURITY_EVENT": "HAZARD",
}
RW_ACTUALS_ALL_30 = RW_ACTUALS_ALL_30.rename(
    columns={k: v for k, v in rename_map_30k.items() if k in RW_ACTUALS_ALL_30.columns}
)

# 2) Add missing column (comment: "Column Missing. Added column in 30K Reports.")
if "IMM_ACTION_TAKEN_RECOM" not in RW_ACTUALS_ALL_30.columns:
    RW_ACTUALS_ALL_30["IMM_ACTION_TAKEN_RECOM"] = pd.NA

# 3) Delete columns marked "Delete from 30k"
drop_cols_30k = [
    "PERSONAL_INJURIES",
    "ACTUAL_WORKDAYS_LTA",
    "COMPANY_NAME",
    "COMPANY_TYPE",
    "CASE_APPROVED_DATE",
    "OUTSIDE_LEGAL_INFLUENCE",
    "RETURNED_TO_WORK_DATE",
]
RW_ACTUALS_ALL_30 = RW_ACTUALS_ALL_30.drop(
    columns=[c for c in drop_cols_30k if c in RW_ACTUALS_ALL_30.columns]
)

# --- Validation ---
print("=" * 60)
print("SCHEMA ALIGNMENT RESULTS")
print("=" * 60)

print("\n[700K] RW_ACTUALS_NH_OBS_700:")
print(f"  Shape: {RW_ACTUALS_NH_OBS_700.shape}")
print(f"  CASE_OCCURENCE_DATE: {'OK' if 'CASE_OCCURENCE_DATE' in RW_ACTUALS_NH_OBS_700.columns else 'MISSING'}")
print(f"  DATETIME: {'STILL PRESENT (unexpected)' if 'DATETIME' in RW_ACTUALS_NH_OBS_700.columns else 'REMOVED (renamed)'}")

print("\n[30K] RW_ACTUALS_ALL_30:")
print(f"  Shape: {RW_ACTUALS_ALL_30.shape}")
for c in ["COMPANY_INVOLVED_TYPE", "HAZARD", "IMM_ACTION_TAKEN_RECOM"]:
    print(f"  {c}: {'OK' if c in RW_ACTUALS_ALL_30.columns else 'MISSING'}")
print("  Columns removed:")
for c in drop_cols_30k:
    print(f"    {c}: {'REMOVED' if c not in RW_ACTUALS_ALL_30.columns else 'STILL PRESENT'}")

SCHEMA ALIGNMENT RESULTS

[700K] RW_ACTUALS_NH_OBS_700:
  Shape: (717041, 36)
  CASE_OCCURENCE_DATE: OK
  DATETIME: REMOVED (renamed)

[30K] RW_ACTUALS_ALL_30:
  Shape: (30609, 36)
  COMPANY_INVOLVED_TYPE: OK
  HAZARD: OK
  IMM_ACTION_TAKEN_RECOM: OK
  Columns removed:
    PERSONAL_INJURIES: REMOVED
    ACTUAL_WORKDAYS_LTA: REMOVED
    COMPANY_NAME: REMOVED
    COMPANY_TYPE: REMOVED
    CASE_APPROVED_DATE: REMOVED
    OUTSIDE_LEGAL_INFLUENCE: REMOVED
    RETURNED_TO_WORK_DATE: REMOVED


## 3. Merge All Four Datasets
Concatenate the four aligned DataFrames (14K, 28K, 30K, 700K) into a single `master_df` and print per-source record counts.

In [37]:
# =============================================================================
# Merge All Four Datasets
# =============================================================================

# Concatenate all four dataframes (14K, 28K, 30K, 700K)
master_df = pd.concat(
    [RW_ACTUALS_PS_14, RW_ACTUALS_ALL_28, RW_ACTUALS_ALL_30, RW_ACTUALS_NH_OBS_700],
    ignore_index=True
)

print(f"Combined master_df (before removing duplicates): {master_df.shape}")
print(f"Total records before deduplication: {len(master_df):,}")

# Per-source breakdown
print(f"\nPer-source record counts:")
print(f"RW_ACTUALS_PS_14:       {len(RW_ACTUALS_PS_14):>10,}")
print(f"RW_ACTUALS_ALL_28:      {len(RW_ACTUALS_ALL_28):>10,}")
print(f"RW_ACTUALS_ALL_30:      {len(RW_ACTUALS_ALL_30):>10,}")
print(f"RW_ACTUALS_NH_OBS_700:  {len(RW_ACTUALS_NH_OBS_700):>10,}")
print(f"{'─'*35}")
print(f"Total:          {len(master_df):>10,}")

print(f"\nTotal columns: {len(master_df.columns)}")
print(f"Columns: {list(master_df.columns)}")


Combined master_df (before removing duplicates): (790405, 36)
Total records before deduplication: 790,405

Per-source record counts:
RW_ACTUALS_PS_14:           14,432
RW_ACTUALS_ALL_28:          28,323
RW_ACTUALS_ALL_30:          30,609
RW_ACTUALS_NH_OBS_700:     717,041
───────────────────────────────────
Total:             790,405

Total columns: 36
Columns: ['CASENO', 'COMPANY', 'FUNCTIONAL_GROUP', 'FUNCTION', 'FUNCTIONAL_AREA', 'FUNCTIONAL_LOCATION', 'FUNCTIONAL_SUB_LOCATION', 'LOCATION_SID', 'LOCATION_SHORT', 'COUNTRY_SHORT', 'SL_COUNTRY', 'SL_LOCATION_LVL_1', 'SL_LOCATION_LVL_2', 'SL_LOCATION_LVL_3', 'SL_LOCATION_LVL_4', 'CASE_OCCURENCE_DATE', 'TITLE', 'COMPANY_INVOLVED_TYPE', 'CASE_TYPE', 'CASE_SEVERITY', 'CASE_DESCRIPTION', 'STATUS', 'HAZARD', 'IMM_ACTION_TAKEN_RECOM', 'FULL_INVESTIGATION_DONE', 'CREATED_DATE', 'MODIFIED_DATE', 'APPROVED_WITHIN_DEADLINE', 'CASE_CLOSED_DATE', 'CASES_NO_OF_REGISTRATIONS', 'VALID_FROM', 'VALID_TO', 'POTENTIAL_SEV_LEVEL', 'RISK_AREA', 'LEARNINGS_A

## 4. Deduplication
Remove duplicate records based on `CASENO`, keeping the entry with the latest `MODIFIED_DATE`.

In [38]:
#=============================================================================
# Remove duplicates in CASENO column, keeping the latest MODIFIED_DATE
#=============================================================================
duplicates_count = master_df['CASENO'].duplicated().sum()
print(f"Duplicates found in CASENO: {duplicates_count}")

# Sort by MODIFIED_DATE descending so the latest entry comes first
master_df['MODIFIED_DATE'] = pd.to_datetime(master_df['MODIFIED_DATE'], errors='coerce')
master_df = master_df.sort_values('MODIFIED_DATE', ascending=False)

# Keep the first occurrence (latest MODIFIED_DATE) of each CASENO
master_df = master_df.drop_duplicates(subset=['CASENO'], keep='first').reset_index(drop=True)

print(f"After removing duplicates: {len(master_df)} records")
print(f"Removed {duplicates_count} duplicate records")

Duplicates found in CASENO: 755479
After removing duplicates: 34926 records
Removed 755479 duplicate records


## 5. Data Cleaning
Remove rows with missing critical text fields (`TITLE`, `CASE_DESCRIPTION`, `CASE_TYPE`, `HAZARD`) and drop completely empty rows.

In [39]:
# ==========================================================================================
# Remove rows with missing critical text fields (TITLE, CASE_DESCRIPTION, CASE_TYPE, HAZARD)
# ==========================================================================================

print(f"Before cleaning: {len(master_df)} records")

# Remove rows with missing critical text fields
required_cols = ['TITLE', 'CASE_DESCRIPTION', 'CASE_TYPE', 'HAZARD']
for col in required_cols:
    if col in master_df.columns:
        before = len(master_df)
        master_df = master_df[master_df[col].notna() & (master_df[col].astype(str).str.strip() != '')]
        print(f"After removing missing {col}: {len(master_df)} records (removed {before - len(master_df)})")

# Remove completely empty rows
master_df = master_df.dropna(how='all')
print(f"\nAfter all cleaning: {len(master_df)} records")

Before cleaning: 34926 records
After removing missing TITLE: 34925 records (removed 1)
After removing missing CASE_DESCRIPTION: 34925 records (removed 0)
After removing missing CASE_TYPE: 34576 records (removed 349)
After removing missing HAZARD: 34576 records (removed 0)

After all cleaning: 34576 records


## 6. EDA — Cases by Case Type
Horizontal bar chart showing the distribution of records across case types (e.g., Process Safety, Near Hit, Observation).

In [40]:
# =============================================================================
# Number of Cases by Case Type
# =============================================================================

if 'CASE_TYPE' in master_df.columns:
    case_counts = master_df['CASE_TYPE'].value_counts().sort_values(ascending=True)

    fig = go.Figure(data=[go.Bar(
        y=case_counts.index.astype(str),
        x=case_counts.values,
        orientation='h',
        text=[f'{v:,}' for v in case_counts.values],
        textposition='outside',
        cliponaxis=False,
        marker=dict(color='#08306b'),
        textfont=dict(size=14)
    )])

    fig.update_layout(
        title=dict(text='Number of Cases by Case Type', font=dict(size=20)),
        xaxis_title=dict(text='Number of Cases', font=dict(size=16)),
        yaxis_title=dict(text='Case Type', font=dict(size=16)),
        xaxis=dict(tickfont=dict(size=14), range=[0, 30000]),
        yaxis=dict(tickfont=dict(size=14)),
        width=1000,
        height=max(500, len(case_counts) * 60),
        showlegend=False,
        hovermode='y unified',
        margin=dict(l=150, r=120, t=60, b=60),
        template='plotly_white'
    )
    fig.show()

    # Save high-resolution PNG for thesis (scale=4 → 4000px wide)
    fig.write_image(PATHS['results'] / 'Number of Cases by Case Type.png', scale=4, engine='kaleido')
    print(f"Saved high-res chart to: {PATHS['results'] / 'Number of Cases by Case Type.png'}")

    print(f"\nUnique case types: {master_df['CASE_TYPE'].nunique()}")
    print(f"Total records: {len(master_df):,}")
    print(f"\n{case_counts.sort_values(ascending=False).to_string()}")
else:
    print("Column 'CASE_TYPE' not found in master_df.")

Saved high-res chart to: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_0/Number of Cases by Case Type.png

Unique case types: 8
Total records: 34,576

CASE_TYPE
Safety                              25164
Process Safety                       4561
Environment                          2768
Asset and Reputation damage/loss     1333
Operational loss                      456
Information Security                  186
Physical Security                     100
Not classified                          8


## 7. EDA — Cases by Country
Horizontal bar chart showing the distribution of records across countries (`SL_COUNTRY`), with missing values filled as "Unknown".

In [41]:
# =============================================================================
# Number of Cases by Country (SL_COUNTRY)
# =============================================================================

if 'SL_COUNTRY' in master_df.columns:
    country_counts = master_df['SL_COUNTRY'].fillna('Unknown').value_counts().sort_values(ascending=True)

    fig = go.Figure(data=[go.Bar(
        y=country_counts.index.astype(str),
        x=country_counts.values,
        orientation='h',
        text=[f'{v:,}' for v in country_counts.values],
        textposition='outside',
        cliponaxis=False,
        marker=dict(color='#08306b'),
        textfont=dict(size=14)
    )])

    fig.update_layout(
        title=dict(text='Number of Cases by Country', font=dict(size=20)),
        xaxis_title=dict(text='Number of Cases', font=dict(size=16)),
        yaxis_title=dict(text='Country', font=dict(size=16)),
        xaxis=dict(tickfont=dict(size=14), range=[0, 30000]),
        yaxis=dict(tickfont=dict(size=14)),
        width=1000,
        height=max(500, len(country_counts) * 60),
        showlegend=False,
        hovermode='y unified',
        margin=dict(l=150, r=120, t=60, b=60),
        template='plotly_white'
    )
    fig.show()

    # Save high-resolution PNG for thesis
    fig.write_image(PATHS['results'] / 'Number of Cases by Country.png', scale=4, engine='kaleido')
    print(f"Saved high-res chart to: {PATHS['results'] / 'Number of Cases by Country.png'}")

    print(f"\nUnique countries: {master_df['SL_COUNTRY'].nunique(dropna=False)}")
    print(f"Total records: {len(master_df):,}")
    print(f"\n{country_counts.sort_values(ascending=False).to_string()}")
else:
    print("Column 'SL_COUNTRY' not found in master_df.")

Saved high-res chart to: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_0/Number of Cases by Country.png

Unique countries: 10
Total records: 34,576

SL_COUNTRY
Germany          14951
Sweden           12224
UK                4673
Netherlands       1990
Hungary            391
Unknown            287
UAE                 29
International       20
Russia              10
Poland               1


## 8. Save Master Dataset
Save the cleaned `master_df` as a single JSON file (`master_df.json`). Any existing file is deleted first to avoid stale data.

In [42]:
# =============================================================================
# Save Master Dataset as JSON
# =============================================================================

output_dir = BASE_DIR / 'Master Dataset 34k'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'master_df.json'
if output_path.exists():
    output_path.unlink()
master_df.to_json(output_path, orient='records', indent=2, force_ascii=False)
print(f"Master dataset saved to: {output_path}")
print(f"Total records: {len(master_df):,}")

Master dataset saved to: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Master Dataset 34k/master_df.json
Total records: 34,576


## 9. Save Per-Country JSON Files
Split `master_df` by `SL_COUNTRY` and save each group as a separate JSON file with `_manual` suffix in `By_SL_Country/`. The folder is cleared first.

In [43]:
# =============================================================================
# Save Separate JSON Files per Country (SL_COUNTRY)
# =============================================================================

import shutil
from tqdm import tqdm

country_dir = BASE_DIR / 'Master Dataset 34k' / 'By_SL_Country'
if country_dir.exists():
    shutil.rmtree(country_dir)
country_dir.mkdir(parents=True, exist_ok=True)

for country, group in master_df.groupby('SL_COUNTRY', dropna=False):
    label = str(country) if pd.notna(country) else 'Unknown'
    safe_name = label.replace('/', '_').replace(' ', '_')
    file_path = country_dir / f'master_df_{safe_name}_manual.json'
    group.to_json(file_path, orient='records', indent=2, force_ascii=False)
    print(f"  {label}: {len(group):,} records → {file_path.name}")

print(f"\nTotal countries: {master_df['SL_COUNTRY'].nunique(dropna=False)}")
print(f"Files saved to: {country_dir}")

  Germany: 14,951 records → master_df_Germany_manual.json
  Hungary: 391 records → master_df_Hungary_manual.json
  International: 20 records → master_df_International_manual.json
  Netherlands: 1,990 records → master_df_Netherlands_manual.json
  Poland: 1 records → master_df_Poland_manual.json
  Russia: 10 records → master_df_Russia_manual.json
  Sweden: 12,224 records → master_df_Sweden_manual.json
  UAE: 29 records → master_df_UAE_manual.json
  UK: 4,673 records → master_df_UK_manual.json
  Unknown: 287 records → master_df_Unknown_manual.json

Total countries: 10
Files saved to: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Master Dataset 34k/By_SL_Country


## 10. Merge English Countries (Manual)
Combine UK, UAE, Russia, and Poland country files into a single `master_df_English_manual.json` — the manually identified English-language subset.

In [44]:
# ==============================================================================================
# Merge UK, UAE, Russia, Poland = master_df_English.json (Manual Processing for English Cases)
# ==============================================================================================

country_dir = BASE_DIR / 'Master Dataset 34k' / 'By_SL_Country'
english_countries = ['UK', 'UAE', 'Russia', 'Poland']

english_dfs = []
for country in english_countries:
    file_path = country_dir / f'master_df_{country}_manual.json'
    df = pd.read_json(file_path)
    english_dfs.append(df)
    print(f"  {country}: {len(df):,} records")

master_df_English = pd.concat(english_dfs, ignore_index=True)
print(f"\nCombined English dataset: {len(master_df_English):,} records")

output_path = country_dir / 'master_df_English_manual.json'
master_df_English.to_json(output_path, orient='records', indent=2, force_ascii=False)
print(f"Saved to: {output_path}")

  UK: 4,673 records
  UAE: 29 records
  Russia: 10 records
  Poland: 1 records

Combined English dataset: 4,713 records
Saved to: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Master Dataset 34k/By_SL_Country/master_df_English_manual.json


## 11. Language Detection — langdetect
Detect the language of each record using `langdetect` (based on `TITLE` + `CASE_DESCRIPTION`). Uses `detect_langs()` with probability filtering to five target languages (English, German, Swedish, Dutch, Hungarian), defaulting to Unknown. Saves per-language JSON files with `_langdetect` suffix.

In [45]:
# =============================================================================
# Language Detection using langdetect (based on TITLE + CASE_DESCRIPTION)
# =============================================================================

from langdetect import detect_langs, LangDetectException

LANG_MAP = {
    'en': 'English',
    'de': 'German',
    'sv': 'Swedish',
    'nl': 'Dutch',
    'hu': 'Hungarian',
}

TARGET_CODES = set(LANG_MAP.keys())

def detect_language_langdetect(text):
    """Detect language of a text string using langdetect, forced to target languages only."""
    text = str(text).strip()
    if not text or text.lower() == 'nan':
        return 'Unknown'
    try:
        results = detect_langs(text)
        # Filter to target languages and pick the highest probability
        for r in results:
            if r.lang in TARGET_CODES:
                return LANG_MAP[r.lang]
        # If no target language detected
        return 'Unknown'
    except LangDetectException:
        return 'Unknown'

# Combine TITLE + CASE_DESCRIPTION for more reliable detection
tqdm.pandas(desc="Detecting language (langdetect)")
master_df['_combined_text'] = (
    master_df['TITLE'].astype(str) + ' ' + master_df['CASE_DESCRIPTION'].astype(str)
)
master_df['LANGUAGE'] = master_df['_combined_text'].progress_apply(detect_language_langdetect)

# Clean up temp column
master_df.drop(columns=['_combined_text'], inplace=True)

print(f"\nLANGUAGE column added (via langdetect):")
print(master_df['LANGUAGE'].value_counts())

# =============================================================================
# Save Separate JSON Files per Detected Language (langdetect)
# =============================================================================

langdetect_dir = BASE_DIR / 'Master Dataset 34k' / 'langdetect'
if langdetect_dir.exists():
    shutil.rmtree(langdetect_dir)
langdetect_dir.mkdir(parents=True, exist_ok=True)

for lang, group in master_df.groupby('LANGUAGE'):
    safe_name = lang.replace('/', '_').replace(' ', '_')
    file_path = langdetect_dir / f'master_df_{safe_name}_langdetect.json'
    group.to_json(file_path, orient='records', indent=2, force_ascii=False)
    print(f"  {lang}: {len(group):,} records → {file_path.name}")

print(f"\nFiles saved to: {langdetect_dir}")

Detecting language (langdetect): 100%|██████████| 34576/34576 [01:12<00:00, 476.37it/s]



LANGUAGE column added (via langdetect):
LANGUAGE
German       15179
Swedish       9789
English       5984
Dutch         1989
Unknown       1278
Hungarian      357
Name: count, dtype: int64
  Dutch: 1,989 records → master_df_Dutch_langdetect.json
  English: 5,984 records → master_df_English_langdetect.json
  German: 15,179 records → master_df_German_langdetect.json
  Hungarian: 357 records → master_df_Hungarian_langdetect.json
  Swedish: 9,789 records → master_df_Swedish_langdetect.json
  Unknown: 1,278 records → master_df_Unknown_langdetect.json

Files saved to: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Master Dataset 34k/langdetect


## 12. Language Detection — lingua
Detect the language of each record using `lingua-py` (based on `TITLE` + `CASE_DESCRIPTION`). Built with a restricted detector for five target languages. Saves per-language JSON files with `_lingua` suffix.

In [46]:
# =============================================================================
# Language Detection using lingua-py (based on TITLE + CASE_DESCRIPTION)
# =============================================================================

from lingua import Language, LanguageDetectorBuilder

# Build detector for the target languages only
TARGET_LANGUAGES = [
    Language.ENGLISH,
    Language.GERMAN,
    Language.SWEDISH,
    Language.DUTCH,
    Language.HUNGARIAN,
]

detector = LanguageDetectorBuilder.from_languages(*TARGET_LANGUAGES).build()

# Mapping from lingua Language enum to readable name
LANG_NAME_MAP = {
    Language.ENGLISH: 'English',
    Language.GERMAN: 'German',
    Language.SWEDISH: 'Swedish',
    Language.DUTCH: 'Dutch',
    Language.HUNGARIAN: 'Hungarian',
}

def detect_language_lingua(text):
    """Detect language of a text string using lingua-py."""
    text = str(text).strip()
    if not text or text.lower() == 'nan':
        return 'Unknown'
    result = detector.detect_language_of(text)
    return LANG_NAME_MAP.get(result, 'Unknown')

# Combine TITLE + CASE_DESCRIPTION for more reliable detection
tqdm.pandas(desc="Detecting language (lingua)")
master_df['_combined_text'] = (
    master_df['TITLE'].astype(str) + ' ' + master_df['CASE_DESCRIPTION'].astype(str)
)
master_df['LANGUAGE'] = master_df['_combined_text'].progress_apply(detect_language_lingua)

# Clean up temp column
master_df.drop(columns=['_combined_text'], inplace=True)

print(f"\nLANGUAGE column added (via lingua-py):")
print(master_df['LANGUAGE'].value_counts())

# =============================================================================
# Save Separate JSON Files per Detected Language (lingua)
# =============================================================================

lingua_dir = BASE_DIR / 'Master Dataset 34k' / 'lingua'
if lingua_dir.exists():
    shutil.rmtree(lingua_dir)
lingua_dir.mkdir(parents=True, exist_ok=True)

for lang, group in master_df.groupby('LANGUAGE'):
    safe_name = lang.replace('/', '_').replace(' ', '_')
    file_path = lingua_dir / f'master_df_{safe_name}_lingua.json'
    group.to_json(file_path, orient='records', indent=2, force_ascii=False)
    print(f"  {lang}: {len(group):,} records → {file_path.name}")

print(f"\nFiles saved to: {lingua_dir}")

Detecting language (lingua): 100%|██████████| 34576/34576 [00:02<00:00, 13645.03it/s]



LANGUAGE column added (via lingua-py):
LANGUAGE
German       15124
Swedish      10837
English       6205
Dutch         2049
Hungarian      361
Name: count, dtype: int64
  Dutch: 2,049 records → master_df_Dutch_lingua.json
  English: 6,205 records → master_df_English_lingua.json
  German: 15,124 records → master_df_German_lingua.json
  Hungarian: 361 records → master_df_Hungarian_lingua.json
  Swedish: 10,837 records → master_df_Swedish_lingua.json

Files saved to: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Master Dataset 34k/lingua


## 13. EDA — Character Count Distribution
Box plots showing the distribution of character counts for Title, Description, and Combined text across all records.

In [47]:
fig.write_image(PATHS['results'] / 'Word Count Distribution.png', scale=6, engine='kaleido')

## 14. EDA — Word Count Distribution
Box plots showing the distribution of word counts for Title, Description, and Combined text across all records.

In [48]:
fig.write_image(PATHS['results'] / 'Word Count Distribution - Title vs Description (Log Scale).png', scale=6, engine='kaleido')

## 15. EDA — Word Count Histogram
Histogram analysis of Title and Description word counts, with a low-word-count impact analysis at thresholds of 3, 5, and 10 words.

In [49]:
# =========================================================================================
# Word Count Distribution - Histogram Analysis (Better than Box Plots for Low Word Counts)
# ==========================================================================================

from plotly.subplots import make_subplots

# Ensure word count columns exist
if 'title_word_count' not in master_df.columns:
    master_df['title_word_count'] = master_df['TITLE'].astype(str).str.split().apply(len)
if 'description_word_count' not in master_df.columns:
    master_df['description_word_count'] = master_df['CASE_DESCRIPTION'].astype(str).str.split().apply(len)
if 'combined_word_count' not in master_df.columns:
    master_df['combined_word_count'] = (master_df['TITLE'].astype(str) + ' ' + master_df['CASE_DESCRIPTION'].astype(str)).str.split().apply(len)

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Title Word Count Distribution', 'Description Word Count Distribution'),
    vertical_spacing=0.18  # Increased spacing to prevent overlap
)

# Title word count histogram
fig.add_trace(go.Histogram(
    x=master_df['title_word_count'],
    nbinsx=50,
    marker_color='#08306b',
    name='Title',
    hovertemplate='Word Count: %{x}<br>Cases: %{y}<extra></extra>',
    opacity=0.85
), row=1, col=1)

# Description word count histogram
fig.add_trace(go.Histogram(
    x=master_df['description_word_count'],
    nbinsx=50,
    marker_color='#41b6c4',  # Different color for distinction
    name='Description',
    hovertemplate='Word Count: %{x}<br>Cases: %{y}<extra></extra>',
    opacity=0.85
), row=2, col=1)

fig.update_xaxes(title_text='Word Count', row=1, col=1)
fig.update_xaxes(title_text='Word Count', row=2, col=1)
fig.update_yaxes(title_text='Number of Cases', row=1, col=1)
fig.update_yaxes(title_text='Number of Cases', row=2, col=1)

fig.update_layout(
    title='Word Count Distribution',
    height=800,  # Increased height for clarity
    showlegend=False,
    template='plotly_white',
    font=dict(size=18, family='Arial'),
    margin=dict(l=80, r=40, t=80, b=60)
)
fig.show()

# Save high-res image to Results/_iteration_0 folder
fig.write_image(PATHS['results'] / 'Word Count Distribution.png', scale=6, engine='kaleido')
print(f"Saved high-res chart to: {PATHS['results'] / 'Word Count Distribution.png'}")

# Low word count analysis - potential model impact
thresholds = [3, 5, 10]
print("=" * 70)
print("LOW WORD COUNT ANALYSIS - Potential Impact on Model Performance")
print("=" * 70)

for t in thresholds:
    title_low = (master_df['title_word_count'] <= t).sum()
    desc_low = (master_df['description_word_count'] <= t).sum()
    combined_low = (master_df['combined_word_count'] <= t).sum()
    print(f"\n  Word count <= {t}:")
    print(f"    Titles:       {title_low:>6,} ({100*title_low/len(master_df):.4f}%)")
    print(f"    Descriptions: {desc_low:>6,} ({100*desc_low/len(master_df):.4f}%)")
    print(f"    Combined:     {combined_low:>6,} ({100*combined_low/len(master_df):.4f}%)")

print("\n" + "=" * 70)
print(f"Total records: {len(master_df):,}")
print("\nNote: Very short texts (<=5 words) provide limited semantic signal for BERT embeddings and may reduce classification accuracy.")

Saved high-res chart to: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_0/Word Count Distribution.png
LOW WORD COUNT ANALYSIS - Potential Impact on Model Performance

  Word count <= 3:
    Titles:        6,374 (18.4348%)
    Descriptions:  3,263 (9.4372%)
    Combined:        267 (0.7722%)

  Word count <= 5:
    Titles:       13,952 (40.3517%)
    Descriptions:  7,189 (20.7919%)
    Combined:      1,230 (3.5574%)

  Word count <= 10:
    Titles:       26,613 (76.9696%)
    Descriptions: 15,432 (44.6321%)
    Combined:      7,288 (21.0782%)

Total records: 34,576

Note: Very short texts (<=5 words) provide limited semantic signal for BERT embeddings and may reduce classification accuracy.


## 16. EDA — Word Count Line Graph (Log Scale)
Line chart comparing Title vs Description word count frequency distributions on a log scale.

In [50]:
# Compute word count columns if missing
if 'title_word_count' not in master_df.columns:
    master_df['title_word_count'] = master_df['TITLE'].astype(str).str.split().apply(len)
if 'description_word_count' not in master_df.columns:
    master_df['description_word_count'] = master_df['CASE_DESCRIPTION'].astype(str).str.split().apply(len)

# Compute frequency distributions
title_freq = master_df['title_word_count'].value_counts().sort_index()
desc_freq = master_df['description_word_count'].value_counts().sort_index()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=title_freq.index,
    y=title_freq.values,
    mode='lines',
    name='Title Word Count',
    line=dict(color='#08306b', width=2),
    hovertemplate='Word Count: %{x}<br>Cases: %{y:,}<extra>Title</extra>'
))

fig.add_trace(go.Scatter(
    x=desc_freq.index,
    y=desc_freq.values,
    mode='lines',
    name='Description Word Count',
    line=dict(color='#006400', width=2),
    hovertemplate='Word Count: %{x}<br>Cases: %{y:,}<extra>Description</extra>'
))

fig.update_layout(
    title='Word Count Distribution - Title vs Description (Log Scale)',
    xaxis_title='Word Count',
    yaxis_title='Number of Cases (Log Scale)',
    yaxis_type='log',
    height=500,
    hovermode='x unified',
    legend=dict(x=0.7, y=0.95)
)
fig.show()

# Save high-res image to Results/_iteration_0 folder
fig.write_image(PATHS['results'] / 'Word Count Distribution - Title vs Description (Log Scale).png', scale=4, engine='kaleido')
print(f"Saved high-res chart to: {PATHS['results'] / 'Word Count Distribution - Title vs Description (Log Scale).png'}")

Saved high-res chart to: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_0/Word Count Distribution - Title vs Description (Log Scale).png


## 17. Load English Datasets
Load the three English-language datasets produced by different methods:
- **Manual**: country-based selection (UK, UAE, Russia, Poland)
- **langdetect**: automated detection via `langdetect`
- **lingua**: automated detection via `lingua-py`

In [51]:
# =============================================================================
# Load English Datasets (Manual, Langdetect, Lingua)
# =============================================================================

master_df_English_manual = pd.read_json(BASE_DIR / 'Master Dataset 34k' / 'By_SL_Country' / 'master_df_English_manual.json')
print(f"English (manual):    {len(master_df_English_manual):,} records")

master_df_English_langdetect = pd.read_json(BASE_DIR / 'Master Dataset 34k' / 'langdetect' / 'master_df_English_langdetect.json')
print(f"English (langdetect): {len(master_df_English_langdetect):,} records")

master_df_English_lingua = pd.read_json(BASE_DIR / 'Master Dataset 34k' / 'lingua' / 'master_df_English_lingua.json')
print(f"English (lingua):    {len(master_df_English_lingua):,} records")

English (manual):    4,713 records
English (langdetect): 5,984 records
English (lingua):    6,205 records


## 18. Binary Classification Formulation
For each of the 3 English datasets, create a binary label (`1` = Process Safety, `0` = Non-Process Safety) and a combined text feature (`TITLE` + `CASE_DESCRIPTION`).

In [52]:
# =============================================================================
# Binary Classification Formulation (for all 3 English datasets)
# =============================================================================

datasets = {
    'manual': master_df_English_manual,
    'langdetect': master_df_English_langdetect,
    'lingua': master_df_English_lingua,
}

for name, df in datasets.items():
    # Create binary label: 1 = Process Safety, 0 = Non-Process Safety
    df['binary_label'] = (df['CASE_TYPE'] == 'Process Safety').astype(int)
    
    # Create combined text feature
    df['text_features'] = (
        df['TITLE'].astype(str) + '. ' + 
        df['CASE_DESCRIPTION'].astype(str)
    )
    
    ps_count = df['binary_label'].sum()
    non_ps_count = (df['binary_label'] == 0).sum()
    print(f"\n[{name}] {len(df):,} records")
    print(f"  Process Safety:     {ps_count:,} ({100*ps_count/len(df):.4f}%)")
    print(f"  Non-Process Safety: {non_ps_count:,} ({100*non_ps_count/len(df):.4f}%)")


[manual] 4,713 records
  Process Safety:     1,409 (29.8960%)
  Non-Process Safety: 3,304 (70.1040%)

[langdetect] 5,984 records
  Process Safety:     1,526 (25.5013%)
  Non-Process Safety: 4,458 (74.4987%)

[lingua] 6,205 records
  Process Safety:     1,522 (24.5286%)
  Non-Process Safety: 4,683 (75.4714%)


## 19. BERT Embedding Generation
Generate BERT-base-uncased `[CLS]` token embeddings for all 3 English datasets. The embeddings folder is cleared before saving. Embeddings and labels are saved as `.npy` files.

In [53]:
# =============================================================================
# BERT Embedding Generation (for all 3 English datasets)
# =============================================================================

# Clear embeddings folder before saving new files
if PATHS['embeddings'].exists():
    shutil.rmtree(PATHS['embeddings'])
    print(f"Cleared: {PATHS['embeddings']}")
PATHS['embeddings'].mkdir(parents=True, exist_ok=True)

from transformers import AutoTokenizer, AutoModel
import torch

# Initialize BERT-base-uncased
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()  # Set to evaluation mode

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Using device: {device}")

def get_bert_embeddings(texts, batch_size=8, max_length=512):
    """Generate BERT [CLS] embeddings for a list of texts."""
    embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating embeddings"):
        batch_texts = texts[i:i + batch_size]
        
        # Tokenize
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)
        
        # Get embeddings
        with torch.no_grad():
            outputs = model(**inputs)
            # Extract [CLS] token embedding (first token)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.extend(cls_embeddings)
    
    return np.array(embeddings)

# Generate and save embeddings for each dataset
embeddings_dict = {}
labels_dict = {}

for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"Generating embeddings for: {name} ({len(df):,} records)")
    print(f"{'='*60}")
    
    texts = df['text_features'].tolist()
    X = get_bert_embeddings(texts)
    y = df['binary_label'].values
    
    # Save embeddings and labels
    np.save(PATHS['embeddings'] / f'bert_embeddings_iteration_0_{name}.npy', X)
    np.save(PATHS['embeddings'] / f'labels_iteration_0_{name}.npy', y)
    
    embeddings_dict[name] = X
    labels_dict[name] = y
    
    print(f"  Embeddings shape: {X.shape}")
    print(f"  Labels shape: {y.shape}")
    print(f"  Saved: bert_embeddings_iteration_0_{name}.npy, labels_iteration_0_{name}.npy")

print(f"\nAll embeddings saved to: {PATHS['embeddings']}")

Cleared: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Datasets/Embeddings/_iteration_0/bert-base-uncased


/Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning:

urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020

/Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/.venv/lib/python3.9/site-packages/huggingface_hub/file_download.py:942: FutureWarning:

`resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.



Using device: cpu

Generating embeddings for: manual (4,713 records)


Generating embeddings: 100%|██████████| 590/590 [02:23<00:00,  4.10it/s]


  Embeddings shape: (4713, 768)
  Labels shape: (4713,)
  Saved: bert_embeddings_iteration_0_manual.npy, labels_iteration_0_manual.npy

Generating embeddings for: langdetect (5,984 records)


Generating embeddings: 100%|██████████| 748/748 [02:57<00:00,  4.21it/s]


  Embeddings shape: (5984, 768)
  Labels shape: (5984,)
  Saved: bert_embeddings_iteration_0_langdetect.npy, labels_iteration_0_langdetect.npy

Generating embeddings for: lingua (6,205 records)


Generating embeddings: 100%|██████████| 776/776 [02:59<00:00,  4.32it/s]


  Embeddings shape: (6205, 768)
  Labels shape: (6205,)
  Saved: bert_embeddings_iteration_0_lingua.npy, labels_iteration_0_lingua.npy

All embeddings saved to: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Datasets/Embeddings/_iteration_0/bert-base-uncased


## 20. Train/Test Split, SVM Training & Evaluation
For each of the 3 datasets:
1. 80/20 stratified train/test split
2. Scale features with `StandardScaler`
3. Train a Linear SVM (`C=1.0`, `class_weight='balanced'`)
4. Evaluate with classification report, confusion matrix, and comparison summary table

In [ ]:
# =============================================================================
# Train/Test Split, SVM Training & Evaluation (for all 3 English datasets)
# =============================================================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

results_summary = []

for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"SVM Training & Evaluation: {name}")
    print(f"{'='*60}")
    
    # Load embeddings and labels
    X = embeddings_dict[name]
    y = labels_dict[name]
    
    # Stratified 80/20 train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)
    
    # Feature scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train Linear SVM
    clf = SVC(kernel='linear', C=1.0, class_weight='balanced', random_state=42)
    clf.fit(X_train_scaled, y_train)
    
    # Predict and evaluate
    y_pred = clf.predict(X_test_scaled)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Non-Process Safety', 'Process Safety']))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    # Collect Macro F1 for summary
    report = classification_report(y_test, y_pred, output_dict=True)
    macro_f1 = report['macro avg']['f1-score']
    results_summary.append({'dataset': name, 'macro_f1': macro_f1})

# Summary Table
summary_df = pd.DataFrame(results_summary)
print("\nSummary Table (Macro F1):")
print(summary_df)

## 21. Summary & Gap Analysis
Aggregate results for all 3 datasets, identify the best-performing dataset by Macro F1, compute the gap to the RQ1 target (≥ 0.85), and save `iteration_0_summary.json`.

In [55]:
    # Save high-resolution PNG for thesis (scale=6 → 6000px wide)
    fig.write_image(PATHS['results'] / 'Number of Cases by Case Type.png', scale=6, engine='kaleido')